# Empirical Analysis of Multi-Layer Perceptrons on Non-Linear Interleaved Manifolds

**Objective:** Demonstrate the capacity of Multi-Layer Perceptrons (MLPs) to synthesize non-linear decision boundaries on topological manifolds where linear classifiers fail.

- **Dataset:** Synthetic two-moons manifold (`sklearn.datasets.make_moons`)
- **Hardware Target:** GPU Acceleration (NVIDIA CUDA) with minimized CPU overhead
- **Methodology:** Stratified Partitioning (70/15/15) with zero-leakage feature standardization

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.utils.seed import set_seed
from src.utils.device import get_compute_device, log_device_telemetry
from src.data.dataset import generate_moons_data, prepare_stratified_splits, scale_features, build_dataloaders
from src.models.architectures import LinearPerceptron, ShallowMLP, DeepModularMLP
from src.training.trainer import ModelTrainer
from src.visualization.plots import plot_decision_boundary, evaluate_decision_grid_gpu

set_seed(42)
device = get_compute_device("cuda")
log_device_telemetry(device)

## 1. Dataset Synthesis & Topological Characterization
We generate $N=3,000$ coordinate samples with Gaussian perturbation $\sigma=0.15$.

In [ ]:
X, y = generate_moons_data(n_samples=3000, noise=0.15, random_state=42)

plt.figure(figsize=(8, 5), dpi=150)
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="#1f77b4", label="Class 0", alpha=0.7, edgecolors="k", s=25)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="#ff7f0e", label="Class 1", alpha=0.7, edgecolors="k", s=25)
plt.title("make_moons Dataset Geometry (sigma = 0.15)", fontsize=12, fontweight="bold")
plt.xlabel("Feature x_1")
plt.ylabel("Feature x_2")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 2. Leakage-Free Stratified Preprocessing
Data is split into Train (70%), Validation (15%), and Test (15%). Scaler is fitted strictly on the train partition.

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = prepare_stratified_splits(
    X, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42
)
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)
train_loader, val_loader, test_loader = build_dataloaders(
    X_train_s, y_train, X_val_s, y_val, X_test_s, y_test, batch_size=64, pin_memory=(device.type == "cuda")
)
print(f"Partition sizes -> Train: {len(X_train_s)}, Val: {len(X_val_s)}, Test: {len(X_test_s)}")

## 3. Model Architecture Construction & GPU Training

In [ ]:
deep_mlp = DeepModularMLP(
    input_dim=2,
    hidden_dims=[64, 32, 16],
    output_dim=1,
    activation="gelu",
    dropout_rate=0.10,
    use_layer_norm=True
).to(device)

trainer = ModelTrainer(
    model=deep_mlp,
    device=device,
    learning_rate=0.005,
    weight_decay=1e-4,
    early_stopping_patience=20
)

history = trainer.fit(train_loader, val_loader, epochs=100, verbose=True)
metrics = trainer.evaluate_test_set(test_loader)
print("\n--- Test Metrics ---")
for k, v in metrics.items():
    if k != "confusion_matrix":
        print(f"{k:<15}: {v:.4f}")

## 4. GPU-Accelerated Decision Boundary Visualization
We evaluate a dense $600 \times 600$ grid ($360,000$ points) directly on CUDA streaming multiprocessors.

In [ ]:
xx, yy, zz = evaluate_decision_grid_gpu(
    model=deep_mlp,
    x_min=X_test_s[:, 0].min() - 0.5,
    x_max=X_test_s[:, 0].max() + 0.5,
    y_min=X_test_s[:, 1].min() - 0.5,
    y_max=X_test_s[:, 1].max() + 0.5,
    resolution=600,
    device=device
)

plt.figure(figsize=(9, 6), dpi=150)
plt.contourf(xx, yy, zz, levels=30, cmap="RdBu_r", alpha=0.85)
plt.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2.0, linestyles="--")
plt.scatter(X_test_s[:, 0], X_test_s[:, 1], c=y_test, cmap=plt.cm.Spectral, edgecolors="k", s=25)
plt.title("Deep Modular MLP: Learned Non-Linear Manifold Separator", fontsize=13, fontweight="bold")
plt.xlabel("Standardized x_1")
plt.ylabel("Standardized x_2")
plt.colorbar(label="P(y=1 | x)")
plt.show()